# Content-Based KNN Music Recommendation System

This notebook builds a **content-based music recommendation system** using **KNN (K-Nearest Neighbors)** with **MinMaxScaler** and **cosine similarity**.

The idea: recommend songs that are **similar** to a song the user already likes, based on audio features like danceability, energy, tempo, valence, and acousticness.

## 1. Import Libraries and Create Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
import joblib

# Example music data with audio features
songs = pd.DataFrame({
    "song_id": [1, 2, 3, 4, 5],
    "title": ["Blinding Lights", "Levitating", "Someone Like You", "Shape of You", "Believer"],
    "artist": ["The Weeknd", "Dua Lipa", "Adele", "Ed Sheeran", "Imagine Dragons"],
    "genre": ["synth-pop", "disco-pop", "pop-ballad", "pop", "rock"],
    "danceability": [0.51, 0.70, 0.56, 0.82, 0.77],
    "energy": [0.73, 0.83, 0.33, 0.65, 0.78],
    "tempo": [171, 103, 135, 96, 125],
    "valence": [0.33, 0.91, 0.28, 0.93, 0.74],
    "acousticness": [0.001, 0.01, 0.89, 0.58, 0.06]
})

songs

,song_id,title,artist,genre,danceability,energy,tempo,valence,acousticness
0,1,Blinding Lights,The Weeknd,synth-pop,0.51,0.73,171,0.33,0.001
1,2,Levitating,Dua Lipa,disco-pop,0.70,0.83,103,0.91,0.010
2,3,Someone Like You,Adele,pop-ballad,0.56,0.33,135,0.28,0.890
3,4,Shape of You,Ed Sheeran,pop,0.82,0.65,96,0.93,0.580
4,5,Believer,Imagine Dragons,rock,0.77,0.78,125,0.74,0.060


## 2. Select and Scale Features with MinMaxScaler

We use **MinMaxScaler** to normalize all features to the **0–1 range**, so that features like `tempo` (90–180) don't dominate distance calculations over features like `danceability` (0–1).

In [2]:
# Select numerical audio features
features = ["danceability", "energy", "tempo", "valence", "acousticness"]
X = songs[features]

# Scale features to 0-1 range using MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Show scaled features
pd.DataFrame(X_scaled, columns=features, index=songs["title"])

,danceability,energy,tempo,valence,acousticness
title,,,,,
Blinding Lights,0.000000,0.80,1.000000,0.076923,0.000000
Levitating,0.612903,1.00,0.093333,0.969231,0.010124
Someone Like You,0.161290,0.00,0.520000,0.000000,1.000000
Shape of You,1.000000,0.64,0.000000,1.000000,0.651294
Believer,0.838710,0.90,0.386667,0.707692,0.066367


## 3. Train KNN Model with Cosine Distance

In [3]:
# Train KNN model with cosine similarity
knn = NearestNeighbors(n_neighbors=4, metric="cosine")
knn.fit(X_scaled)
print("KNN model trained successfully!")

KNN model trained successfully!


## 4. Build Recommendation Function

In [4]:
def recommend_songs(song_title, songs, model, X_scaled, top_n=3):
    """Recommend similar songs based on audio features using KNN."""
    # Find the song index
    song_idx = songs[songs["title"].str.lower() == song_title.lower()].index

    if len(song_idx) == 0:
        return f"Song '{song_title}' not found."

    song_idx = song_idx[0]

    # Find nearest neighbors
    distances, indices = model.kneighbors(
        [X_scaled[song_idx]],
        n_neighbors=top_n + 1
    )

    recommendations = []
    for i, distance in zip(indices[0], distances[0]):
        # Skip the input song itself
        if i == song_idx:
            continue
        recommendations.append({
            "title": songs.iloc[i]["title"],
            "artist": songs.iloc[i]["artist"],
            "similarity_score": round(1 - distance, 4)
        })

    return pd.DataFrame(recommendations)

## 5. Test Recommendations

In [5]:
# Test with "Blinding Lights"
print("=== Recommendations for 'Blinding Lights' ===")
result1 = recommend_songs("Blinding Lights", songs, knn, X_scaled)
display(result1)

# Test with "Shape of You"
print("\n=== Recommendations for 'Shape of You' ===")
result2 = recommend_songs("Shape of You", songs, knn, X_scaled)
display(result2)

# Test with "Someone Like You"
print("\n=== Recommendations for 'Someone Like You' ===")
result3 = recommend_songs("Someone Like You", songs, knn, X_scaled)
display(result3)

=== Recommendations for 'Blinding Lights' ===


,title,artist,similarity_score
0,Believer,Imagine Dragons,0.6146
1,Levitating,Dua Lipa,0.4949
2,Someone Like You,Adele,0.3560



=== Recommendations for 'Shape of You' ===


,title,artist,similarity_score
0,Believer,Imagine Dragons,0.8737
1,Levitating,Dua Lipa,0.8685
2,Someone Like You,Adele,0.4239



=== Recommendations for 'Someone Like You' ===


,title,artist,similarity_score
0,Shape of You,Ed Sheeran,0.4239
1,Blinding Lights,The Weeknd,0.3560
2,Believer,Imagine Dragons,0.2402


## 6. Add Genre with One-Hot Encoding

Including genre information makes songs from similar genres more likely to be recommended together.

In [6]:
# One-hot encode the genre column
songs_encoded = pd.get_dummies(songs, columns=["genre"], drop_first=False)
print("Columns after encoding:", list(songs_encoded.columns))

# New feature set including genre columns
genre_cols = [c for c in songs_encoded.columns if c.startswith("genre_")]
features_with_genre = features + genre_cols
print("Features used:", features_with_genre)

X_genre = songs_encoded[features_with_genre]

# Re-scale with MinMaxScaler
scaler_genre = MinMaxScaler()
X_genre_scaled = scaler_genre.fit_transform(X_genre)

# Re-train KNN
knn_genre = NearestNeighbors(n_neighbors=4, metric="cosine")
knn_genre.fit(X_genre_scaled)

# Compare recommendations
print("\n=== Without genre (Blinding Lights) ===")
display(recommend_songs("Blinding Lights", songs, knn, X_scaled))

print("\n=== With genre (Blinding Lights) ===")
display(recommend_songs("Blinding Lights", songs_encoded, knn_genre, X_genre_scaled))

Columns after encoding: ['song_id', 'title', 'artist', 'danceability', 'energy', 'tempo', 'valence', 'acousticness', 'genre_disco-pop', 'genre_pop', 'genre_pop-ballad', 'genre_rock', 'genre_synth-pop']
Features used: ['danceability', 'energy', 'tempo', 'valence', 'acousticness', 'genre_disco-pop', 'genre_pop', 'genre_pop-ballad', 'genre_rock', 'genre_synth-pop']

=== Without genre (Blinding Lights) ===


,title,artist,similarity_score
0,Believer,Imagine Dragons,0.6146
1,Levitating,Dua Lipa,0.4949
2,Someone Like You,Adele,0.3560



=== With genre (Blinding Lights) ===


,title,artist,similarity_score
0,Believer,Imagine Dragons,0.4010
1,Levitating,Dua Lipa,0.3264
2,Someone Like You,Adele,0.2110


## 7. Compare Distance Metrics

Let's compare **cosine**, **euclidean**, and **manhattan** distance metrics side by side.

In [7]:
query_song = "Blinding Lights"

for metric in ["cosine", "euclidean", "manhattan"]:
    knn_temp = NearestNeighbors(n_neighbors=4, metric=metric)
    knn_temp.fit(X_scaled)
    result = recommend_songs(query_song, songs, knn_temp, X_scaled)
    print(f"\n=== Metric: {metric} ===")
    display(result)


=== Metric: cosine ===


,title,artist,similarity_score
0,Believer,Imagine Dragons,0.6146
1,Levitating,Dua Lipa,0.4949
2,Someone Like You,Adele,0.3560



=== Metric: euclidean ===


,title,artist,similarity_score
0,Believer,Imagine Dragons,-0.2214
1,Someone Like You,Adele,-0.3793
2,Levitating,Dua Lipa,-0.4262



=== Metric: manhattan ===


,title,artist,similarity_score
0,Believer,Imagine Dragons,-1.2492
1,Someone Like You,Adele,-1.5182
2,Levitating,Dua Lipa,-1.6220


## 8. Save and Load the Model

In [8]:
# Save the model and scaler
joblib.dump(knn, "knn_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Model and scaler saved!")

# Load and verify
knn_loaded = joblib.load("knn_model.pkl")
scaler_loaded = joblib.load("scaler.pkl")

print("\n=== Verification with loaded model ===")
display(recommend_songs("Blinding Lights", songs, knn_loaded, X_scaled))

Model and scaler saved!

=== Verification with loaded model ===


,title,artist,similarity_score
0,Believer,Imagine Dragons,0.6146
1,Levitating,Dua Lipa,0.4949
2,Someone Like You,Adele,0.3560


## 9. Simple FastAPI Endpoint

Below is a minimal FastAPI app. To run it, save the code to a `.py` file and run `uvicorn app:app --reload`.

In [9]:
fastapi_code = '''
from fastapi import FastAPI
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

app = FastAPI()

# Load model and data
knn = joblib.load("knn_model.pkl")
scaler = joblib.load("scaler.pkl")

songs = pd.DataFrame({
    "song_id": [1, 2, 3, 4, 5],
    "title": ["Blinding Lights", "Levitating", "Someone Like You", "Shape of You", "Believer"],
    "artist": ["The Weeknd", "Dua Lipa", "Adele", "Ed Sheeran", "Imagine Dragons"],
    "danceability": [0.51, 0.70, 0.56, 0.82, 0.77],
    "energy": [0.73, 0.83, 0.33, 0.65, 0.78],
    "tempo": [171, 103, 135, 96, 125],
    "valence": [0.33, 0.91, 0.28, 0.93, 0.74],
    "acousticness": [0.001, 0.01, 0.89, 0.58, 0.06]
})

features = ["danceability", "energy", "tempo", "valence", "acousticness"]
X_scaled = scaler.transform(songs[features])

def recommend_songs(song_title, top_n=5):
    song_idx = songs[songs["title"].str.lower() == song_title.lower()].index
    if len(song_idx) == 0:
        return []
    song_idx = song_idx[0]
    distances, indices = knn.kneighbors([X_scaled[song_idx]], n_neighbors=top_n + 1)
    results = []
    for i, d in zip(indices[0], distances[0]):
        if i == song_idx:
            continue
        results.append({"title": songs.iloc[i]["title"], "artist": songs.iloc[i]["artist"], "similarity": round(1 - d, 4)})
    return results

@app.get("/recommend")
def recommend(song_title: str):
    return recommend_songs(song_title)
'''

print(fastapi_code)
print("\\n# Save this to app.py and run: uvicorn app:app --reload")
print("# Then visit: http://127.0.0.1:8000/recommend?song_title=Blinding%20Lights")


from fastapi import FastAPI
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

app = FastAPI()

# Load model and data
knn = joblib.load("knn_model.pkl")
scaler = joblib.load("scaler.pkl")

songs = pd.DataFrame({
    "song_id": [1, 2, 3, 4, 5],
    "title": ["Blinding Lights", "Levitating", "Someone Like You", "Shape of You", "Believer"],
    "artist": ["The Weeknd", "Dua Lipa", "Adele", "Ed Sheeran", "Imagine Dragons"],
    "danceability": [0.51, 0.70, 0.56, 0.82, 0.77],
    "energy": [0.73, 0.83, 0.33, 0.65, 0.78],
    "tempo": [171, 103, 135, 96, 125],
    "valence": [0.33, 0.91, 0.28, 0.93, 0.74],
    "acousticness": [0.001, 0.01, 0.89, 0.58, 0.06]
})

features = ["danceability", "energy", "tempo", "valence", "acousticness"]
X_scaled = scaler.transform(songs[features])

def recommend_songs(song_title, top_n=5):
    song_idx = songs[songs["title"].str.lower() == song_title.lower()].index
    if len(song_idx) == 0:
        return []
    song_idx = song_